# Train the Easy Game Engine Builder model

Trains **Qwen3 4B** on examples made from the engine (`train.jsonl`), checks it, converts it for the browser (WebLLM), and saves it to your Google Drive.

**Before running:**
1. Put `train.jsonl` and `eval.jsonl` (from `ai-training/data/` in the project) in **My Drive / EGE-AI /**.
2. Runtime → Change runtime type → **L4** (or **A100**, faster). Turn on **High-RAM** if offered.
3. Runtime → **Run all**. Say yes when it asks to use your Google Drive.

Takes about 4–5 hours on an A100 for ~38,000 examples. It saves progress to Drive: if Colab disconnects, run all again and it carries on.


In [ ]:
# 1. Settings (change only if you want to)
BASE_MODEL = "unsloth/Qwen3-4B"          # the WebLLM browser library for Qwen3 4B is reused, so keep the 4B
DRIVE_DIR  = "/content/drive/MyDrive/EGE-AI"
EPOCHS     = 1                             # ~38,000 examples: one pass is enough (about 4-5 hours on an A100)
MAX_LEN    = 4096                          # the assistant's prompts fit in this
MODEL_NAME = "EGE-Builder-4B-v2"         # a new name for each training round (browsers keep old ones by name)
CHECKPOINTS = f"{DRIVE_DIR}/checkpoints-{MODEL_NAME}"


In [ ]:
# 2. Your Google Drive (the examples come from there, the model goes there)
from google.colab import drive
drive.mount('/content/drive')
import os
assert os.path.exists(f"{DRIVE_DIR}/train.jsonl"), f"Put train.jsonl in {DRIVE_DIR} first"
os.makedirs(CHECKPOINTS, exist_ok=True)
!wc -l {DRIVE_DIR}/train.jsonl {DRIVE_DIR}/eval.jsonl


In [ ]:
# 3. Training tools
!pip -q install --upgrade unsloth trl datasets


In [ ]:
# 4. The model, with LoRA (only small extra layers are trained: fast, fits the GPU)
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)


In [ ]:
# 5. The examples, written exactly as the assistant sees them (Qwen3 chat format, thinking off:
#    the assistant answers its build steps directly)
from datasets import load_dataset
data = load_dataset("json", data_files={"train": f"{DRIVE_DIR}/train.jsonl", "eval": f"{DRIVE_DIR}/eval.jsonl"})
def to_text(row):
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False, enable_thinking=False)}
data = data.map(to_text, remove_columns=data["train"].column_names)
print(data)
print(data["train"][0]["text"][-600:])


In [ ]:
# 6. Train (saves to Drive every 200 steps; running again carries on from there)
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=data["train"], eval_dataset=data["eval"],
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_LEN,
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        num_train_epochs=EPOCHS, learning_rate=2e-4, warmup_ratio=0.03, lr_scheduler_type="cosine",
        logging_steps=20, eval_strategy="steps", eval_steps=200,
        save_strategy="steps", save_steps=200, save_total_limit=2,
        output_dir=CHECKPOINTS, report_to="none", bf16=True,
    ),
)
# Learn only the answers, not the (long) instructions.
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
import glob
resume = bool(glob.glob(f"{CHECKPOINTS}/checkpoint-*"))
trainer.train(resume_from_checkpoint=resume)


In [ ]:
# 7. Try it on a few test requests it never saw
import json, random
FastLanguageModel.for_inference(model)
tests = [json.loads(l) for l in open(f"{DRIVE_DIR}/eval.jsonl")]
for t in random.sample(tests, 3):
    msgs = t["messages"][:-1]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False, return_tensors="pt").to("cuda")
    out = model.generate(ids, max_new_tokens=400, temperature=0.2, do_sample=True)
    print("REQUEST:", msgs[-1]["content"][-200:])
    print("MODEL:  ", tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)[:600])
    print("WANTED: ", t["messages"][-1]["content"][:600], "\n")


In [ ]:
%%writefile /content/convert_to_webllm.py
# 8. The browser converter (the same as ai-training/convert_to_webllm.py in the project). Run this cell; it only saves the file.
"""Converts a merged (16-bit) Hugging Face Qwen3 model to WebLLM's q4f16_1 weights.

Writes params_shard_*.bin and tensor-cache.json in the same layout as mlc-ai/Qwen3-4B-q4f16_1-MLC, so
the prebuilt Qwen3 4B browser library runs it. It does what `mlc_llm convert_weight --quantization q4f16_1`
does, with only numpy (that command can crash while compiling its quantize step).

  python convert_to_webllm.py <merged model folder> <output folder>

The output folder also needs mlc-chat-config.json and the tokenizer files: `mlc_llm gen_config` makes those.
"""
import hashlib
import json
import os
import struct
import sys

import numpy as np

GROUP = 32          # values per scale
MAX_INT = 7         # int4 values are stored as 0..14, 7 meaning zero
PER_WORD = 8        # int4 values per uint32
SHARD_BYTES = 32 * 1024 * 1024
ROWS_AT_ONCE = 8192


def open_weights(folder):
    """Every tensor in the folder's .safetensors files: name -> (file, dtype, shape, start, end)."""
    tensors = {}
    for file in sorted(f for f in os.listdir(folder) if f.endswith(".safetensors")):
        path = os.path.join(folder, file)
        with open(path, "rb") as f:
            size = struct.unpack("<Q", f.read(8))[0]
            header = json.loads(f.read(size))
        for name, t in header.items():
            if name != "__metadata__":
                start, end = t["data_offsets"]
                tensors[name] = (path, t["dtype"], t["shape"], 8 + size + start, 8 + size + end)
    return tensors


def read_f16(entry):
    """A tensor as float16, the way MLC reads it (bfloat16 goes through float32)."""
    path, dtype, shape, start, end = entry
    raw = np.memmap(path, dtype=np.uint8, mode="r", offset=start, shape=(end - start,))
    if dtype == "BF16":
        f32 = (raw.view(np.uint16).astype(np.uint32) << 16).view(np.float32)
    elif dtype == "F16":
        return np.array(raw.view(np.float16)).reshape(shape)
    elif dtype == "F32":
        f32 = raw.view(np.float32)
    else:
        raise ValueError(f"Unsupported weight type {dtype}")
    return f32.astype(np.float16).reshape(shape)


def quantize(w):
    """q4f16_1: groups of 32 along each row, scale = max |w| / 7, values packed 8 per uint32 (lowest bits first).

    Each step rounds to float16 like MLC's float16 kernel (which multiplies by 1/7 and rounds halves up), so the
    result is bit-for-bit what `mlc_llm convert_weight` writes."""
    rows, cols = w.shape
    assert cols % GROUP == 0, f"row length {cols} is not a multiple of {GROUP}"
    q_weight = np.empty((rows, cols // PER_WORD), np.uint32)
    q_scale = np.empty((rows, cols // GROUP), np.float16)
    inv_max = np.float32(np.float16(1 / MAX_INT))
    for r in range(0, rows, ROWS_AT_ONCE):
        g = w[r:r + ROWS_AT_ONCE].reshape(-1, cols // GROUP, GROUP)
        scale = (np.abs(g).max(-1).astype(np.float32) * inv_max).astype(np.float16)
        x = (g.astype(np.float32) / scale.astype(np.float32)[..., None]).astype(np.float16)
        x = (x.astype(np.float32) + np.float32(MAX_INT)).astype(np.float16)
        q = np.clip(np.floor(x.astype(np.float32) + 0.5), 0, 2 * MAX_INT).astype(np.uint32)
        q = q.reshape(q.shape[0], cols // PER_WORD, PER_WORD) << (np.arange(PER_WORD, dtype=np.uint32) * 4)
        q_weight[r:r + ROWS_AT_ONCE] = np.bitwise_or.reduce(q, axis=-1)
        q_scale[r:r + ROWS_AT_ONCE] = scale
    return q_weight, q_scale


def browser_tensors(weights, config):
    """(name, array) for every tensor the Qwen3 browser model loads, with MLC's joined layers."""
    get = lambda name: read_f16(weights[name])

    def quantized(name, w):
        q_weight, q_scale = quantize(w)
        yield f"{name}.q_weight", q_weight
        yield f"{name}.q_scale", q_scale

    yield from quantized("model.embed_tokens", get("model.embed_tokens.weight"))
    for i in range(config["num_hidden_layers"]):
        p = f"model.layers.{i}"
        yield f"{p}.input_layernorm.weight", get(f"{p}.input_layernorm.weight")
        yield f"{p}.post_attention_layernorm.weight", get(f"{p}.post_attention_layernorm.weight")
        yield f"{p}.self_attn.q_norm.weight", get(f"{p}.self_attn.q_norm.weight")
        yield f"{p}.self_attn.k_norm.weight", get(f"{p}.self_attn.k_norm.weight")
        qkv = np.concatenate([get(f"{p}.self_attn.{x}_proj.weight") for x in "qkv"])
        yield from quantized(f"{p}.self_attn.c_attn", qkv)
        yield from quantized(f"{p}.self_attn.o_proj", get(f"{p}.self_attn.o_proj.weight"))
        gate_up = np.concatenate([get(f"{p}.mlp.gate_proj.weight"), get(f"{p}.mlp.up_proj.weight")])
        yield from quantized(f"{p}.mlp.gate_up_proj", gate_up)
        yield from quantized(f"{p}.mlp.down_proj", get(f"{p}.mlp.down_proj.weight"))
    yield "model.norm.weight", get("model.norm.weight")
    if not config.get("tie_word_embeddings", True):
        yield from quantized("lm_head", get("lm_head.weight"))


def convert(src, out):
    config = json.load(open(os.path.join(src, "config.json")))
    if config.get("model_type") != "qwen3":
        raise ValueError(f"Expected a qwen3 model, got {config.get('model_type')}")
    weights = open_weights(src)
    os.makedirs(out, exist_ok=True)
    shards, shard, shard_bytes = [], [], 0

    def flush():
        nonlocal shard, shard_bytes
        if not shard:
            return
        path = f"params_shard_{len(shards)}.bin"
        md5 = hashlib.md5()
        records, offset = [], 0
        with open(os.path.join(out, path), "wb") as f:
            for name, a in shard:
                data = np.ascontiguousarray(a).tobytes()
                f.write(data)
                md5.update(data)
                records.append({"name": name, "shape": list(a.shape), "dtype": str(a.dtype), "format": "f32-to-bf16",
                                "nbytes": len(data), "byteOffset": offset})
                offset += len(data)
        shards.append({"dataPath": path, "format": "raw-shard", "nbytes": offset, "records": records,
                       "md5sum": md5.hexdigest()})
        shard, shard_bytes = [], 0

    count = 0
    for name, a in browser_tensors(weights, config):
        if shard and shard_bytes + a.nbytes > SHARD_BYTES:
            flush()
        shard.append((name, a))
        shard_bytes += a.nbytes
        count += 1
        print(f"\r{count} tensors", end="", flush=True)
    flush()
    print()

    total = sum(s["nbytes"] for s in shards)
    source_params = sum(int(np.prod(t[2])) for n, t in weights.items()
                        if not (n == "lm_head.weight" and config.get("tie_word_embeddings", True)))
    meta = {"ParamSize": count, "ParamBytes": float(total), "BitsPerParam": total * 8 / source_params}
    with open(os.path.join(out, "tensor-cache.json"), "w") as f:
        json.dump({"metadata": meta, "records": shards}, f, indent=2)
    print(f"{count} tensors, {len(shards)} shards, {total / 2**30:.2f} GB -> {out}")


if __name__ == "__main__":
    if len(sys.argv) != 3:
        sys.exit(__doc__)
    convert(sys.argv[1], sys.argv[2])

In [ ]:
# 9. Save the full trained model (merged), then convert it for the browser (WebLLM)
#    If Colab disconnected after training: run cells 1, 2, 3, 8, then this one. It picks the trained model up from Drive.
import glob, json, os
if "model" not in globals():
    from unsloth import FastLanguageModel
    last = max(glob.glob(f"{CHECKPOINTS}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
    print("Loading the trained model from", last)
    model, tokenizer = FastLanguageModel.from_pretrained(last, max_seq_length=MAX_LEN, load_in_4bit=False)
if not os.path.exists("/content/merged/config.json"):
    model.save_pretrained_merged("/content/merged", tokenizer, save_method="merged_16bit")
# Newer Transformers saves rope_theta inside "rope_parameters"; MLC wants it at the top level
cfg_path = "/content/merged/config.json"
cfg = json.load(open(cfg_path))
cfg.setdefault("rope_theta", (cfg.get("rope_parameters") or {}).get("rope_theta", 1000000))
json.dump(cfg, open(cfg_path, "w"), indent=2)
OUT = f"/content/{MODEL_NAME}-q4f16_1-MLC"
# MLC writes the settings and tokenizer files (its own weight converter crashes, so cell 8's converter does the weights)
!pip -q install --pre -f https://mlc.ai/wheels mlc-llm-nightly-cpu mlc-ai-nightly-cpu
!python -m mlc_llm gen_config /content/merged --quantization q4f16_1 --conv-template qwen3 --context-window-size 40960 --prefill-chunk-size 2048 -o {OUT}
!python /content/convert_to_webllm.py /content/merged {OUT}
assert os.path.exists(f"{OUT}/mlc-chat-config.json") and os.path.exists(f"{OUT}/tensor-cache.json"), "Converting failed: send the error above to Claude"
!ls {OUT} | head; ls {OUT} | wc -l

In [ ]:
# 10. Copy the browser model to your Drive: download this folder and add it to the engine
import shutil
dest = f"{DRIVE_DIR}/{MODEL_NAME}-q4f16_1-MLC"
shutil.rmtree(dest, ignore_errors=True)
shutil.copytree(OUT, dest)
print(f"Done. In Google Drive: EGE-AI/{MODEL_NAME}-q4f16_1-MLC  ->  download it, then in the project run:")
print(f"  npm run pack:ai -- --local ~/Downloads/{MODEL_NAME}-q4f16_1-MLC")

In [ ]:
# 11. Download the finished model to this computer automatically (it's also saved in Google Drive)
from google.colab import files
zip_path = shutil.make_archive(f"/content/{MODEL_NAME}-q4f16_1-MLC", "zip", OUT)
print("Downloading", zip_path)
files.download(zip_path)